### Ingestión del archivo "genre.csv"

In [0]:
%run "../includes/configuration"


In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment","")
v_environment = dbutils.widgets.get("p_environment")


### Paso 1 - Leer el archivo CSV usando "DataFrameReader" de Spark
### La documentación para los comandos de Spark la sacamos de https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/index.html
### Este notebook está basado en el 01-Ingestion File Movie, ya que tenemos que hacer lo mismo para el genre.csv, sin embargo aca lo haremos más directo y no voy a comentar, en el notebook 01 hay pasos extras informativos y esta todo comentado

In [0]:
genre_schema = "genreId INT, genreName STRING"






In [0]:
#genre_df = spark.read.option("header",True).schema(genre_schema).csv("abfss://bronze@moviee.dfs.core.windows.net/genre.csv")

genre_df = spark.read.option("header",True).schema(genre_schema).csv(f"{bronze_folder_path}/genre.csv")



In [0]:
genre_df.printSchema()

In [0]:
display(genre_df)

### Paso 2 - Cambiar el nombre de las columnas según lo requerido

In [0]:
genre_renamed_of = genre_df \
                        .withColumnRenamed("genreId", "genre_id") \
                        .withColumnRenamed("genreName", "genre_name")



                        



In [0]:
display(genre_renamed_of)

### Paso 3 - Agregar la columnas "ingestion_date" y "environment" al DataFrame

In [0]:
from pyspark.sql.functions import current_timestamp, lit











In [0]:
#genre_final_df = genre_renamed_of.withColumn("ingestion_date", current_timestamp()) \
#                                    .withColumn("env", lit("production"))

genre_final_df = add_ingestion_date(genre_renamed_of) \
                    .withColumn("environment", lit(v_environment))                                    
                                  











                                    

In [0]:
display(genre_final_df)

### Paso 4 - Escribir datos en el datalake en formato "Parquet"

In [0]:
#genre_final_df.write.mode("overwrite").parquet("abfss://silver@moviee.dfs.core.windows.net/genres")

genre_final_df.write.mode("overwrite").parquet(f"{silver_folder_path}/genres")










In [0]:
#display(spark.read.parquet("abfss://silver@moviee.dfs.core.windows.net/genres"))

display(spark.read.parquet(f"{silver_folder_path}/genres"))

### Paso 5 - Escribir datos en una managed table en el contenedor silver

#### Esto es un cambio posterior en el notebook, pasaría a reemplazar al paso 4, ya que ahora no quiero más generar archivos de salida en la capa silver a partir del dataframe, sino meter esa info en una tabla administrada por spark, pero dejo igual lo del paso 4, ya que esto se crea dentro de la carpeta _unitystorage y no pisa lo del paso 4

In [0]:
genre_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.genres")

In [0]:
%sql
SELECT * FROM movie_silver.genres

In [0]:
dbutils.notebook.exit("Success")